### Kevin original part - user input student id

In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import pytesseract
import re
import os
import datetime
import numpy as np
import csv

# Uncomment this line and set correct path if Tesseract not found automatically
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

current_image = None

def clean_student_id(ocr_id):
    return (
        ocr_id.upper()
        .replace("O9", "09")
        .replace("O", "0", 1)
        .replace("I", "1")
        .replace("S", "5")
    )

def extract_name_and_id(image_pil):
    # Convert to OpenCV and grayscale
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

    # Preprocessing
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    thresh = cv2.adaptiveThreshold(
        sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY, 15, 10
    )

    # OCR
    text = pytesseract.image_to_string(thresh)
    print("OCR Result:\n", text)

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''

    for i, line in enumerate(lines):
        # Normalize and search for ID-like string
        line_cleaned = line.replace("O", "0").replace("S", "5")
        match = re.search(r'\d{2}[A-Z0-9]{3}\d{5}', line_cleaned)
        if match:
            raw_id = match.group()
            student_id = raw_id.replace("0", "O", 1)  # fix back one likely 'O'

            # Debug: print nearby lines
            print("\nNearby lines for name detection:")
            for j in range(max(0, i - 3), i):
                if j < len(lines):
                    print(f"→ {lines[j]}")

            # Look for valid name above
            for j in range(max(0, i - 3), i):
                candidate = lines[j]
                if (
                    not re.search(r'\d', candidate)
                    and len(candidate.split()) >= 2
                    and candidate.isupper()
                    and not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])
                ):
                    name = candidate
                    break
            break

    return name.strip(), student_id.strip()

def show_confirm_buttons():
    confirm_btn.pack(pady=5)
    retake_btn.pack(pady=5)

def hide_confirm_buttons():
    confirm_btn.pack_forget()
    retake_btn.pack_forget()

def display_image_pil(pil_img):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((300, 200))
    img_tk = ImageTk.PhotoImage(img_resized)
    panel.config(image=img_tk)
    panel.image = img_tk
    show_confirm_buttons()

def upload_image():
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        display_image_pil(pil_img)

def take_picture():
    cap = cv2.VideoCapture(0)

    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 480, 300
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2

    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press 's' to capture, 'q' to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow("Capture ID Card", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        display_image_pil(pil_img)

def confirm_image():
    global current_image

    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    name, student_id = extract_name_and_id(current_image)

    if not name and not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    # Editable popup
    edit_window = tk.Toplevel(root)
    edit_window.title("Confirm Student Info")

    tk.Label(edit_window, text="Name:").grid(row=0, column=0, padx=10, pady=5, sticky="e")
    name_entry = tk.Entry(edit_window, width=40)
    name_entry.insert(0, name)
    name_entry.grid(row=0, column=1, padx=10, pady=5)

    tk.Label(edit_window, text="Student ID:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    id_entry = tk.Entry(edit_window, width=40)
    corrected_id = clean_student_id(student_id)
    id_entry.insert(0, corrected_id)
    id_entry.grid(row=1, column=1, padx=10, pady=5)

    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()

        if not final_name or not final_id:
            messagebox.showerror("Error", "Both Name and Student ID are required.")
            return

        folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
        os.makedirs(folder_name, exist_ok=True)

        img_filename = f"{final_name}.{final_id}.jpg"
        img_path = os.path.join(folder_name, img_filename)
        current_image.save(img_path)

        # Save to CSV
        csv_file = "student_records.csv"
        header = ["Name", "Student ID", "Image Path"]
        row = [final_name.replace("_", " "), final_id, img_path]

        file_exists = os.path.isfile(csv_file)
        with open(csv_file, "a", newline="") as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(header)
            writer.writerow(row)

        messagebox.showinfo("Saved", f"Student image saved to:\n{img_path}")
        hide_confirm_buttons()
        edit_window.destroy()

    tk.Button(edit_window, text="✅ Confirm & Save", command=save_data).grid(row=2, column=0, columnspan=2, pady=10)

def retake_or_reselect():
    hide_confirm_buttons()
    panel.config(image=None)
    panel.image = None
    global current_image
    current_image = None

# GUI setup
root = tk.Tk()
root.title("ID Card Image Capture")

upload_btn = tk.Button(root, text="Upload Image", command=upload_image)
upload_btn.pack(pady=10)

capture_btn = tk.Button(root, text="Take a Picture", command=take_picture)
capture_btn.pack(pady=10)

panel = tk.Label(root)
panel.pack(padx=10, pady=10)

confirm_btn = tk.Button(root, text="✅ Confirm", command=confirm_image)
retake_btn = tk.Button(root, text="🔁 Retake/Reselect", command=retake_or_reselect)

root.mainloop()


## ziwen combine with kevin part
### registration phase (user input student id & email + face detection)
i updated the flow and UI to make it more pretty and easy to use 

In [47]:
import cv2
import pytesseract
import face_recognition
import re
import os
import csv
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import messagebox, filedialog
from tkinter import ttk

# Configure tesseract path (update this if needed)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Global current image
current_image = None

# OCR & ID CLEANUP FUNCTIONS 
def clean_student_id(ocr_id):
    return (
        ocr_id.upper()
        .replace("O9", "09")
        .replace("O", "0", 1)
        .replace("I", "1")
        .replace("S", "5")
    )

def extract_name_and_id(image_pil):
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    thresh = cv2.adaptiveThreshold(sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY, 15, 10)

    text = pytesseract.image_to_string(thresh)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''

    for i, line in enumerate(lines):
        line_cleaned = line.replace("O", "0").replace("S", "5")
        match = re.search(r'\d{2}[A-Z0-9]{3}\d{5}', line_cleaned)
        if match:
            raw_id = match.group()
            student_id = raw_id.replace("0", "O", 1)
            for j in range(max(0, i - 3), i):
                candidate = lines[j]
                if (not re.search(r'\d', candidate)
                    and len(candidate.split()) >= 2
                    and candidate.isupper()
                    and not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])):
                    name = candidate
                    break
            break

    return name.strip(), student_id.strip()

# GUI FUNCTIONS 
def display_image_pil(pil_img):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((350, 250))
    img_tk = ImageTk.PhotoImage(img_resized)
    panel.config(image=img_tk)
    panel.image = img_tk
    confirm_btn.pack(pady=10)
    retake_btn.pack()
    upload_btn.pack_forget()
    capture_btn.pack_forget()

def upload_image():
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        display_image_pil(pil_img)

def take_picture():
    cap = cv2.VideoCapture(0)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 480, 300
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2
    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press SPACE to capture, ESC to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow("Capture ID Card", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 32:  # Spacebar
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        display_image_pil(pil_img)

def confirm_image():
    global current_image
    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    name, student_id = extract_name_and_id(current_image)
    if not name and not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    # Editable popup
    edit_window = tk.Toplevel(root)
    edit_window.title("Confirm Student Info")

    ttk.Label(edit_window, text="Please confirm or edit your information below:").grid(row=0, column=0, columnspan=2, pady=(10, 5))

    def force_uppercase_name(*args):
        current = name_var.get()
        name_var.set(current.upper())

    name_var = tk.StringVar()
    name_var.trace_add("write", force_uppercase_name)

    ttk.Label(edit_window, text="Name:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    name_entry = ttk.Entry(edit_window, width=40, textvariable=name_var)
    name_var.set(name)
    name_entry.grid(row=1, column=1, padx=10, pady=5)


    ttk.Label(edit_window, text="Student ID:").grid(row=2, column=0, padx=10, pady=5, sticky="e")
    id_entry = ttk.Entry(edit_window, width=40)
    corrected_id = clean_student_id(student_id)
    id_entry.insert(0, corrected_id)
    id_entry.grid(row=2, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="School Email:").grid(row=3, column=0, padx=10, pady=5, sticky="e")
    email_entry = ttk.Entry(edit_window, width=40)
    email_entry.grid(row=3, column=1, padx=10, pady=5)

    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()
        email = email_entry.get().strip()
        if not final_name or not final_id or not email:
            messagebox.showerror("Error", "Name, Student ID, and Email are required.")
            return

        folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
        os.makedirs(folder_name, exist_ok=True)
        img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
        current_image.save(img_path)

        with open("student_records.csv", "a", newline="") as f:
            writer = csv.writer(f)
            if os.stat("student_records.csv").st_size == 0:
                writer.writerow(["Name", "Student ID", "Email", "Image Path"])
            writer.writerow([final_name.replace("_", " "), final_id, email, img_path])

        messagebox.showinfo("Next Step", "Now we will capture your face using the webcam.")

        # FACE CAPTURE  
        cap = cv2.VideoCapture(0)
        from mtcnn import MTCNN
        detector = MTCNN()
        captured = False

        while not captured:
            ret, frame = cap.read()
            if not ret:
                continue
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = detector.detect_faces(rgb)
            cv2.putText(frame, "Center your face and press SPACE to capture, ESC to cancel", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.imshow("Live Face Capture", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == 32 and results:  # Spacebar
                x, y, w, h = results[0]['box']
                top, right, bottom, left = y, x + w, y + h, x
                face_img_path = os.path.join(folder_name, "centerface.jpg")
                Image.fromarray(rgb[y:y+h, x:x+w]).save(face_img_path)
                encodings = face_recognition.face_encodings(rgb, known_face_locations=[(top, right, bottom, left)])
                if encodings:
                    np.save(os.path.join(folder_name, "face_encoding.npy"), encodings[0])
                    messagebox.showinfo("Success", "Registration complete! You may now register the next student.")
                    captured = True
                else:
                    messagebox.showwarning("Encoding Failed", "Face encoding failed. Try again with better lighting or angle.")

            elif key == 27:
                break

        cap.release()
        cv2.destroyAllWindows()
        edit_window.destroy()

        # Reset UI for next student
        retake_or_reselect()

    ttk.Button(edit_window, text="Confirm & Save", command=save_data).grid(row=4, column=0, columnspan=2, pady=10)

def retake_or_reselect():
    global current_image
    panel.config(image=None)
    panel.image = None
    current_image = None
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
    upload_btn.pack(pady=10)
    capture_btn.pack(pady=5)

# GUI SETUP  
root = tk.Tk()
root.title("🎓 Convocation Registration System")
root.geometry("480x600")

welcome = ttk.Label(root, text="Welcome to Convocation Registration System!", font=("Helvetica", 14, "bold"))
welcome.pack(pady=10)

instruction = ttk.Label(root, text="Please register using your student ID card (upload or webcam)")
instruction.pack()

upload_btn = ttk.Button(root, text="Upload Student ID Card", command=upload_image)
upload_btn.pack(pady=10)

capture_btn = ttk.Button(root, text="Capture Student ID via Webcam", command=take_picture)
capture_btn.pack(pady=5)

panel = ttk.Label(root)
panel.pack(padx=10, pady=10)

confirm_btn = ttk.Button(root, text="Confirm ID Card", command=confirm_image)
retake_btn = ttk.Button(root, text="Retake / Select Another Image", command=retake_or_reselect)

root.mainloop()


### ceremony day (face verfication)

In [ ]:
import cv2
import numpy as np
import face_recognition
import os

# Step 1: Load all registered student encodings
student_encodings = []
student_labels = []
folder_base = "StudentidFolder"

for folder in os.listdir(folder_base):
    folder_path = os.path.join(folder_base, folder)
    encoding_path = os.path.join(folder_path, "face_encoding.npy")
    if os.path.exists(encoding_path):
        encoding = np.load(encoding_path)
        student_encodings.append(encoding)
        student_labels.append(folder)  # Keep raw format for name-ID split

if not student_encodings:
    print("=== No registered student encodings found. ===")
    exit()

# Step 2: Start webcam
cap = cv2.VideoCapture(0)
print("📷 Webcam running. Stand in front of the camera to be recognized.")

font = cv2.FONT_HERSHEY_SIMPLEX
recognized_label = ""

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    face_locations = face_recognition.face_locations(rgb)

    for (top, right, bottom, left) in face_locations:
        live_encoding = face_recognition.face_encodings(rgb, [ (top, right, bottom, left) ])[0]
        distances = face_recognition.face_distance(student_encodings, live_encoding)
        best_match_index = np.argmin(distances)
        best_distance = distances[best_match_index]

        if best_distance < 0.6:
            matched_raw = student_labels[best_match_index]
            if "." in matched_raw:
                name, sid = matched_raw.split(".", 1)
                display_text = f"{name.replace('_', ' ')} - {sid}"
            else:
                display_text = matched_raw.replace("_", " ")
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.putText(frame, f"{display_text}", (left, top - 10), font, 0.8, (0, 255, 0), 2)
        else:
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
            cv2.putText(frame, "Unknown Face", (left, top - 10), font, 0.8, (0, 0, 255), 2)

    cv2.putText(frame, "Press ESC to quit", (10, 30), font, 0.6, (255, 255, 255), 2)
    cv2.imshow("Graduation Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
